In [1]:
import pathlib
import pickle

import mcr_py.helper_functions
import mcr_py.mcr.data
import mcr_py.mcr.path
import mcr_py.mcr5.labels
import mcr_py.minute_city.minute_city
import mcr_py.utils.strtime
import numpy as np
import pandas as pd
import polars as pl
from mcr_py.mcr.path import GTFSPath, Path, PathType
from mcr_py.utils.logger import setup

setup("INFO")

In [2]:
city_name = "cologne"
date = "20250926"

In [3]:
data_directory = pathlib.Path("../data/")
base_directory = data_directory / date
cache_path = base_directory / "cache/"
osm_path = base_directory / "osm_raw"
geometa_path = base_directory / f"cache/{city_name}_geometa.json"
mcr5_output_path = base_directory / f"mcr5_results/{city_name}_reduced_paths"
mcr5_output_path_comp = base_directory / f"mcr5_results/{city_name}"
gtfs_clean_dir = base_directory / f"gtfs_clean/{city_name}/"
gtfs_clean_struct = gtfs_clean_dir / "structs.pkl"
gtfs_clean_stops = gtfs_clean_dir / "stops.parquet"
geo_meta, geo_data = mcr_py.helper_functions.load_auxiliary_classes(
    geo_meta_path=geometa_path,
    city_id="Koeln",
    osm_path=osm_path,
    cache_path=cache_path,
)

[19:52:57] INFO     Loading OSM walking                               ]8;id=967138;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=897093;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
[19:53:00] INFO     Loading OSM walking done (3.06 seconds)           ]8;id=29355;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=906450;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
           INFO     Loading OSM POIs                                  ]8;id=283026;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=284937;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loading OSM POIs done (0.01 seconds)              ]8;id=728909;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=677248;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loadi

In [4]:
with open(mcr5_output_path / "public_transport_0" / "891fa199c77ffff.pkl", "rb") as f:
    hex = pickle.load(f)

In [ ]:
labels = pd.DataFrame(
    [
        (label.node_id, label.values[0], label.values[1], n_transfers, label)
        for n_transfers, bags in hex["bags_i"].items()
        for bag in bags.values()
        for label in bag
    ],
    columns=["osm_node_id", "time", "cost", "n_transfers", "label"],
)
labels["human_time"] = labels["time"].apply(
    lambda x: mcr_py.utils.strtime.seconds_to_str_time(x)
)

In [6]:
labels = labels.merge(
    geo_data.pois.select(
        pl.col("nearest_osm_node").alias("osm_node_id").cast(pl.Int64),
        pl.col("lat").alias("poi_lat"),
        pl.col("long").alias("poi_long"),
        "poi_type",
    ).to_pandas(),
    how="left",
    on="osm_node_id",
)

In [7]:
labels["duplicate"] = labels.duplicated(
    subset=["osm_node_id", "time", "cost", "poi_type", "poi_lat", "poi_long"], keep=False
)

In [8]:
labels[labels.cost > 0]

,osm_node_id,time,cost,n_transfers,label,human_time,poi_lat,poi_long,poi_type,duplicate
987,1817930821,292909,220,0,"IntermediateLabel(values=[292909, 220], hidden...",08:08:10,NaN,NaN,NaN,True
988,4903558504,292814,220,0,"IntermediateLabel(values=[292814, 220], hidden...",08:08:01,NaN,NaN,NaN,True
989,8500540440,292851,220,0,"IntermediateLabel(values=[292851, 220], hidden...",08:08:05,NaN,NaN,NaN,True
990,52155672,293024,220,0,"IntermediateLabel(values=[293024, 220], hidden...",08:08:22,NaN,NaN,NaN,True
991,5971324660,292637,220,0,"IntermediateLabel(values=[292637, 220], hidden...",08:07:43,50.955561,6.921746,Shops,True
...,...,...,...,...,...,...,...,...,...,...
2026,12681689250,293043,220,1,"IntermediateLabel(values=[293043, 220], hidden...",08:08:24,NaN,NaN,NaN,True
2039,8503465355,292800,220,1,"IntermediateLabel(values=[292800, 220], hidden...",08:08:00,NaN,NaN,NaN,True
2044,4903558506,292887,220,1,"IntermediateLabel(values=[292887, 220], hidden...",08:08:08,NaN,NaN,NaN,True
2074,2415878778,292636,220,1,"IntermediateLabel(values=[292636, 220], hidden...",08:07:43,NaN,NaN,NaN,True


In [9]:
nodes = geo_data.osm_nodes.with_columns(pl.col("osm_id").alias("id")).to_pandas()
nodes_pl = geo_data.osm_nodes.with_columns(pl.col("osm_id").alias("id"))

In [10]:
path_manager = hex["path_manager"]

In [11]:
from mcr_py.mcr.data import NetworkType

translator_map = {
    PathType.WALKING: dict(
        geo_data.osm_nodes.select(pl.col("rx_node_id").alias("osm"), "osm_id").rows()
    ),
    PathType.CYCLING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.CYCLING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.DRIVING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.DRIVING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.PUBLIC_TRANSPORT: None,
}
stops_by_id = pl.read_parquet(gtfs_clean_stops).to_pandas().set_index("stop_id", drop=True)

In [ ]:
def format_meta(meta, previous_meta, start_time) -> str:
    values = meta["values"]
    arrival_time = values[0]
    cost = values[1]

    if previous_meta:
        previous_values = previous_meta["values"]
        previous_arrival_time = previous_values[0]
        previous_cost = previous_values[1]

        arrival_time -= previous_arrival_time
        cost -= previous_cost
    else:
        arrival_time -= start_time

    return f"{mcr_py.utils.strtime.seconds_to_str_time(arrival_time)} ({cost})"

In [13]:
color_map = {
    "Shops": "orange",
    "Grocery": "red",
    "Parks": "green",
    "Education": "blue",
    "Banks": "violet",
    "Health": "darkgreen",
    "Sustenance": "yellow",
}

mode_color = {
    PathType.WALKING: "grey",
    PathType.CYCLING_WALKING: "blue",
    PathType.DRIVING_WALKING: "violet",
    PathType.PUBLIC_TRANSPORT: "green",
}

In [14]:
import rich.console

console = rich.console.Console()

In [15]:
import plotly.graph_objects as go
from mcr_py.mcr.label import IntermediateLabel


def plot_paths_on_map(
    labels: pd.DataFrame,
    nodes: pd.DataFrame,
    path_manager: mcr_py.mcr.path.PathManager,
    translator_map: dict[PathType, dict[str, int]],
    color_map: dict[str, str],
    path_name: str,
) -> None:
    nodes_by_id = nodes.set_index("id", drop=False)
    fig = go.Figure()

    walking_paths_long = []
    walking_paths_lat = []
    cycling_paths_long = []
    cycling_paths_lat = []
    pt_paths_long = []
    pt_paths_lat = []
    poi_assoc_long = []
    poi_assoc_lat = []

    for row in labels.itertuples():
        label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
        end_node_id = row.osm_node_id
        end_node = nodes_by_id.loc[end_node_id]

        paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
            path_manager.paths, label, translator_map
        )
        for i, path in enumerate(paths):
            if isinstance(path, Path):
                if path.path == []:
                    continue
                if path.path_type == PathType.WALKING:
                    walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                    path_lat = [node.lat for node in walking_path_nodes]
                    path_lon = [node.long for node in walking_path_nodes]
                    if i + 1 == len(paths):
                        path_lat.append(end_node.lat)  # type: ignore
                        path_lon.append(end_node.long)  # type: ignore
                    else:
                        pass
                        # path_lat.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].lat)
                        # path_lon.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].long)
                    walking_paths_long.extend(path_lon)
                    walking_paths_long.append(None)
                    walking_paths_lat.extend(path_lat)
                    walking_paths_lat.append(None)
                if path.path_type in [PathType.DRIVING_WALKING, PathType.CYCLING_WALKING]:
                    path_nodes = [
                        nodes_by_id.loc[int(node_id[1:])]  # pyright: ignore[reportIndexIssue]
                        for node_id in path.path
                        if node_id[0] == "D"  # pyright: ignore[reportIndexIssue]
                    ]
                    path_lat = [node.lat for node in path_nodes]
                    path_lon = [node.long for node in path_nodes]
                    cycling_paths_long.extend(path_lon)
                    cycling_paths_long.append(None)
                    cycling_paths_lat.extend(path_lat)
                    cycling_paths_lat.append(None)
            elif isinstance(path, GTFSPath):
                start_stop_id = path.start_stop_id
                end_stop_id = path.end_stop_id
                start_stop = stops_by_id.loc[str(start_stop_id)]
                end_stop = stops_by_id.loc[str(end_stop_id)]
                trip = path.trip_id
                if len(trip) >= 10:
                    trip = trip[:10] + "..."

                path_lat = [node.stop_lat for node in [start_stop, end_stop]]
                path_lon = [node.stop_lon for node in [start_stop, end_stop]]
                pt_paths_long.extend(path_lon)
                pt_paths_long.append(None)
                pt_paths_lat.extend(path_lat)
                pt_paths_lat.append(None)
        if row.poi_type is not np.nan:
            poi_assoc_long.append(row.poi_long)
            poi_assoc_long.append(end_node.long)
            poi_assoc_lat.append(row.poi_lat)
            poi_assoc_lat.append(end_node.lat)
            poi_assoc_long.append(None)
            poi_assoc_lat.append(None)

    fig.add_trace(
        go.Scattermap(
            lon=walking_paths_long,
            lat=walking_paths_lat,
            mode="lines",
            marker={"size": 1, "color": "grey"},
            legendgroup="Walking",
            name="Walking",
            showlegend=True,
        )
    )
    fig.add_trace(
        go.Scattermap(
            lon=cycling_paths_long,
            lat=cycling_paths_lat,
            mode="lines",
            marker={"size": 1, "color": "blue"},
            legendgroup="Bicycle",
            name="Bicycle",
            showlegend=True,
        )
    )
    fig.add_trace(
        go.Scattermap(
            lon=pt_paths_long,
            lat=pt_paths_lat,
            mode="lines",
            marker={"size": 1, "color": "green"},
            legendgroup="Public Transport",
            name="Public Transport",
            showlegend=True,
        )
    )

    nodes_long = [nodes_by_id.loc[row.osm_node_id].long for row in labels.itertuples()]
    nodes_lat = [nodes_by_id.loc[row.osm_node_id].lat for row in labels.itertuples()]
    fig.add_trace(
        go.Scattermap(
            lon=nodes_long,
            lat=nodes_lat,
            mode="markers",
            marker={"size": 8, "color": "grey"},
            name="Node",
            legendgroup="Nodes",
            showlegend=True,
        )
    )

    fig.add_trace(
        go.Scattermap(
            lon=poi_assoc_long,
            lat=poi_assoc_lat,
            mode="lines",
            marker={"size": 8, "color": "grey"},  # pyright: ignore[reportArgumentType]
            showlegend=False,
        )
    )

    for poi_type in labels.poi_type.dropna().unique():
        labels_with_poi_type = labels[labels["poi_type"] == poi_type]
        poi_long = [
            row.poi_long
            for row in labels_with_poi_type.itertuples()
            if row.poi_type == poi_type
        ]
        poi_lat = [
            row.poi_lat
            for row in labels_with_poi_type.itertuples()
            if row.poi_type == poi_type
        ]
        fig.add_trace(
            go.Scattermap(
                lon=poi_long,
                lat=poi_lat,
                mode="markers",
                marker={"size": 10, "color": color_map[poi_type]},
                name=poi_type,
                legendgroup=poi_type,
                showlegend=True,
            )
        )

    fig.add_trace(
        go.Scattermap(
            lat=["50.948884"],  # Latitude of the marker
            lon=["6.917342"],  # Longitude of the marker
            mode="markers",
            marker={"size": 14, "color": "lightgreen"},
            text=["Starting Point"],  # Hover text
            name="Starting Point",
            showlegend=True,
            legendgroup="Starting Point",
        )
    )

    fig.update_layout(
        map={
            "style": "carto-positron-nolabels",
            "zoom": 16,  # street level
            "center": {"lat": 50.948884, "lon": 6.917342},
        },
        legend={
            "title": {
                "text": "Transport & POI Types",  # your legend title
                "font": {"size": 14, "color": "black"},
            },
            "orientation": "v",  # vertical (default) or "h" for horizontal
            "x": 1,  # horizontal position (0=left, 1=right)
            "y": 1,  # vertical position (0=bottom, 1=top)
        },
        margin={"r": 0, "t": 0, "l": 0, "b": 0},
        height=2000,
        width=2400,
    )
    fig.write_image(path_name, scale=2)
    fig.show()

In [16]:
plot_paths_on_map(
    labels=labels,
    nodes=nodes,
    path_manager=path_manager,
    translator_map=translator_map,
    color_map=color_map,
    path_name="../figures/mcr5/cologne_reduced_20250926/public_transport_ehrenfeld.png",
)

[19:53:15] INFO     Chromium init'ed with kwargs {}              ]8;id=517192;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=60053;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#182\182]8;;\
           INFO     Found chromium path: /usr/bin/chromium       ]8;id=458539;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=384719;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#209\209]8;;\
           INFO     Temp directory created: /tmp/tmpt90m3hkg.     ]8;id=21930;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=479479;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/utils/_tmpfile.py#80\80]8;;\
          

In [17]:
from mcr_py.structs import build
from mcr_py.utils import storage

structs_dict = storage.read_any_dict(gtfs_clean_struct)
(
    # stop_times_by_trip,
    trip_ids_by_route,
    stops_by_route,
    idx_by_stop_by_route,
    routes_by_stop,
    times_by_stop_by_trip,
    stop_id_set,
    # route_id_set,
    # trip_id_set,
) = build.unpack_structs(structs_dict)

In [18]:
df = pl.read_parquet(gtfs_clean_dir / "trips.parquet")
df_stops = pl.read_parquet(gtfs_clean_dir / "stops.parquet")

In [19]:
from mcr_py import add_nearest_node_to_df

df_stops = add_nearest_node_to_df(
    df_stops.with_columns(
        pl.col("stop_lat").alias("lat"),
        pl.col("stop_lon").alias("long"),
    ),
    nodes_pl,
    4839,
)

In [20]:
valid_stops = [
    "68807",
    "263434",
    "555954",
    "13457",
    "43938",
    "243120",
    "173432",
    "7336",
    "181059",
    "560399",
    "241497",
    "252020",
    "467413",
]

In [21]:
filtered = df_stops.filter(pl.col("stop_id").is_in(valid_stops))
filtered_osm_nodes = filtered.select(pl.col("nearest_node_osm_id")).to_series().to_list()
stop_to_arrival = dict(
    filtered.to_pandas()
    .merge(labels, left_on="nearest_node_osm_id", right_on="osm_node_id")[
        ["stop_id", "human_time"]
    ]
    .values
)

In [ ]:
valid_routes = []
routes_to_stop_ids = {}
routes_to_osm_arrival_time = {}
routes_to_arrival_time_at_stop = {}
for stop in valid_stops:
    valid_routes.extend(routes_by_stop[stop])
    for route_id in routes_by_stop[stop]:
        if route_id not in routes_to_stop_ids:
            routes_to_stop_ids[route_id] = []
        routes_to_stop_ids[route_id].append(stop)
        if route_id not in routes_to_osm_arrival_time:
            routes_to_osm_arrival_time[route_id] = []
        routes_to_osm_arrival_time[route_id].append(stop_to_arrival.get(stop))
        if route_id not in routes_to_arrival_time_at_stop:
            routes_to_arrival_time_at_stop[route_id] = []
        for _trip in trip_ids_by_route[route_id]:
            if (
                times_by_stop_by_trip[_trip][stop][1] > 287000
                and times_by_stop_by_trip[_trip][stop][1] < 297000
            ):
                routes_to_arrival_time_at_stop[route_id].append(
                    stop
                    + " - "
                    + mcr_py.utils.strtime.seconds_to_str_time(
                        times_by_stop_by_trip[_trip][stop][1]
                    )
                )

In [23]:
df.filter((pl.col("route_id").is_in(valid_routes))).with_columns(
    pl.col("route_id")
    .replace_strict(routes_to_stop_ids, return_dtype=pl.List)
    .alias("related_stops"),
    pl.col("route_id")
    .replace_strict(routes_to_osm_arrival_time, return_dtype=pl.List)
    .alias("related_times"),
    pl.col("route_id")
    .replace_strict(routes_to_arrival_time_at_stop, return_dtype=pl.List)
    .alias("trip_departure_times"),
).filter(
    (pl.col("trip_departure_times").list.len() > 0)
    & pl.col("related_times").list.first().is_not_null()
).sort(by=pl.col("related_times").list.min()).head(10)

route_id,service_id,trip_id,old_route_id,first_stop_id,trip_departure_time,related_stops,related_times,trip_departure_times
str,i64,i64,i64,str,str,list[str],list[str],list[str]
"""16623_16""",1473,109826,16623,"""456120""","""08:01:00""","[""243120""]","[""08:04:58""]","[""243120 - 08:14:00""]"
"""16623_17""",1723,111684,16623,"""456120""","""07:52:00""","[""243120""]","[""08:04:58""]","[""243120 - 08:05:00""]"
"""15356_10""",2687,45761,15356,"""223882""","""07:54:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 08:07:00"", ""43938 - 08:06:00""]"
"""15356_27""",1473,133442,15356,"""223882""","""07:46:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 07:59:00""]"
"""15356_189""",1723,861430,15356,"""223882""","""07:50:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 08:03:00"", ""43938 - 08:02:00""]"
"""15356_356""",1723,1528044,15356,"""223882""","""08:00:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 08:13:00"", ""43938 - 08:12:00""]"
"""19833_119""",1723,496950,19833,"""344014""","""07:51:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 07:59:00""]"
"""19833_135""",2687,569080,19833,"""344014""","""08:06:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 08:14:00"", ""43938 - 08:13:00""]"
"""19833_218""",2687,978832,19833,"""344014""","""07:51:00""","[""68807"", ""43938""]","[""08:06:53"", ""08:05:16""]","[""68807 - 07:59:00""]"
